# **Rendra Gunawan**

In [1]:
%%writefile requirements.txt
unsloth @ git+https://github.com/unslothai/unsloth.git
unsloth_zoo
torch
torchao
trl
peft
cut_cross_entropy
transformers
datasets
huggingface_hub
hf_transfer
msgspec
bitsandbytes
xformers
accelerate
sentencepiece
protobuf
pypdf
gradio
ipywidgets
google-generativeai

Writing requirements.txt


In [2]:
!pip install -r requirements.txt

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-bwp5wjgq/unsloth_55cc740faab042b699bc9d0a9fca6507
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-bwp5wjgq/unsloth_55cc740faab042b699bc9d0a9fca6507
  Resolved https://github.com/unslothai/unsloth.git to commit 7c63bc8c4f18c1d0b0eec5e656ae20797893b500
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 114.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 114.2 MB/s eta 0:00:00
   

In [3]:
import os
import re
import json
import torch
import random
from google.colab import drive
from pypdf import PdfReader
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1551: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [6]:
from google.colab import drive
# Mount Google Drive
drive.mount('/content/drive')


Mounted at /content/drive


In [7]:
import os
# Buat direktori penyimpanan jika belum ada
output_dir = "/content/drive/MyDrive/junior-ai-engineer/data"
os.makedirs(output_dir, exist_ok=True)

In [8]:
from pypdf import PdfReader
pdf_path = "/content/drive/MyDrive/junior-ai-engineer/data/permenkes-no-10-tahun-2024.pdf"
reader = PdfReader(pdf_path)

# **Preprocessing**

In [15]:
import json
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Buat direktori penyimpanan jika belum ada
output_dir = "/content/drive/MyDrive/junior-ai-engineer/data"
os.makedirs(output_dir, exist_ok=True)

from pypdf import PdfReader

pdf_path = f"{output_dir}/permenkes-no-10-tahun-2024.pdf"
jsonl_output_path = f"{output_dir}/raw_text.jsonl"

reader = PdfReader(pdf_path)

# 3. Ekstraksi Teks dan Simpan Langsung ke File JSONL
with open(jsonl_output_path, "w", encoding="utf-8") as f:
    for page_num, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = text.strip()

        # Buat struktur data per halaman
        data = {"page": page_num, "text": text}

        # Tulis sebagai satu baris JSON (JSONL)
        f.write(json.dumps(data, ensure_ascii=False) + "\n")

print(f"Ekstraksi selesai! File JSONL tersimpan di: {jsonl_output_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ekstraksi selesai! File JSONL tersimpan di: /content/drive/MyDrive/junior-ai-engineer/data/raw_text.jsonl


In [17]:
import json
import os
import re

jsonl_raw_path = f"{output_dir}/raw_text.jsonl"
jsonl_cleaned_path = f"{output_dir}/cleaned_text.jsonl"


# 1. Fungsi Pembersihan Teks
def clean_text(raw_text):
  cleaned_text = raw_text

  # 1. Menghapus nomor halaman yang tidak diperlukan (misal: -1-, -12-)
  cleaned_text = re.sub(r'-\s*\d+\s*-', '', cleaned_text)

  # 2. Menghapus OCR/encoding noise
  cleaned_text = re.sub(r'[ŒДѼЖ]', '', cleaned_text)

  # 3. Menghapus spasi dan tab berlebih
  cleaned_text = re.sub(r'[ \t]+', ' ', cleaned_text)

  # 4. Menghapus spasi di awal dan akhir baris
  cleaned_text = re.sub(r' *\n *', '\n', cleaned_text)

  # 5. Menghapus spasi berlebih
  cleaned_text = re.sub(r' {2,}', ' ', cleaned_text)

  return cleaned_text.strip()


# 2. Proses Pembersihan Teks dari raw_text.jsonl ke cleaned_text.jsonl
with (
    open(jsonl_raw_path, 'r', encoding='utf-8') as f_in,
    open(jsonl_cleaned_path, 'w', encoding='utf-8') as f_out,
):
  for line in f_in:
    data = json.loads(line)

    # Ambil teks mentah per halaman
    raw_text = data.get('text', '')

    # Bersihkan teks
    cleaned_page_text = clean_text(raw_text)

    # Buat dictionary baru dengan teks yang sudah dibersihkan
    cleaned_data = {'page': data.get('page'), 'text': cleaned_page_text}

    # Tulis per baris ke format JSONL
    f_out.write(json.dumps(cleaned_data, ensure_ascii=False) + '\n')

print(
    f'Proses pembersihan selesai! File tersimpan di: {jsonl_cleaned_path}\n'
)

Proses pembersihan selesai! File tersimpan di: /content/drive/MyDrive/junior-ai-engineer/data/cleaned_text.jsonl



In [18]:
import json

jsonl_raw_path = f"{output_dir}/raw_text.jsonl"
jsonl_cleaned_path = f"{output_dir}/cleaned_text.jsonl"

# 1. Baca seluruh teks dari raw_text.jsonl
raw_text = ""
with open(jsonl_raw_path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        raw_text += data.get("text", "")

# 2. Baca seluruh teks dari cleaned_text.jsonl
cleaned_text = ""
with open(jsonl_cleaned_path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        cleaned_text += data.get("text", "")

# 3. Tampilkan hasil perbandingan jumlah karakter
print(f"Jumlah Karakter Raw    : {len(raw_text)}")
print(f"Jumlah Karakter Cleaned: {len(cleaned_text)}")
print(f"Karakter Terhapus      : {len(raw_text) - len(cleaned_text)}")

Jumlah Karakter Raw    : 12313
Jumlah Karakter Cleaned: 11872
Karakter Terhapus      : 441


# **Modeling**

In [28]:
import os
import json
from datasets import load_dataset
from unsloth import FastLanguageModel

# 1. Inisialisasi Model & Tokenizer Unsloth
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# 2. Persiapan Adapter PEFT / LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 3. Path File cleaned_text.jsonl (Perbaikan Variabel Path)
output_dir = "/content/drive/MyDrive/junior-ai-engineer/data"
train_jsonl_path = os.path.join(output_dir, "cleaned_text.jsonl")  # Didefinisikan di sini

# 4. Memuat Dataset JSONL
dataset_train = load_dataset("json", data_files={"train": train_jsonl_path}, split="train")

# 5. Format Prompt Alpaca yang Disesuaikan dengan Struktur cleaned_text.jsonl
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def format_prompts(examples):
    instruction_text = "Pelajari dokumen berikut untuk memahami isi Peraturan Kesehatan Permenkes No 10 Tahun 2024:"
    texts = []

    # Menyesuaikan pembacaan key "text" dan "page" dari cleaned_text.jsonl
    for text_content, page_num in zip(examples["text"], examples["page"]):
        input_content = f"Halaman {page_num}:\n{text_content}"
        output_content = "Dokumen berhasil dipelajari."

        # Format ke Alpaca Prompt + EOS Token
        formatted_text = alpaca_prompt.format(instruction_text, input_content, output_content) + tokenizer.eos_token
        texts.append(formatted_text)

    return {"formatted_text": texts}

# Aplikasikan fungsi pemformatan ke dataset
dataset_train = dataset_train.map(format_prompts, batched=True)

# 6. Cek Pratinjau Data Pertama
print(f"Total baris/halaman berhasil dimuat: {len(dataset_train)}")
print("\n--- Contoh Sample Data Pertama yang Siap Masuk Trainer ---")
print(dataset_train[0]["formatted_text"][:600])

==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-1B-bnb-4bit as a legacy tokenizer.
Unsloth 2026.9.4 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


Map:   0%|          | 0/7 [00:00<?, ? examples/s]

Total baris/halaman berhasil dimuat: 7

--- Contoh Sample Data Pertama yang Siap Masuk Trainer ---
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Pelajari dokumen berikut untuk memahami isi Peraturan Kesehatan Permenkes No 10 Tahun 2024:

### Input:
Halaman 1:
PERATURAN MENTERI KESEHATAN REPUBLIK INDONESIA
NOMOR 10 TAHUN 2024
TENTANG
JARINGAN DOKUMENTASI DAN INFORMASI HUKUM
DI LINGKUNGAN KEMENTERIAN KESEHATAN

DENGAN RAHMAT TUHAN YANG MAHA ESA

MENTERI KESEHATAN REPUBLIK INDONESIA,


Menimbang : a. bahwa untuk meningkatkan pelayanan kepada
masyarakat terhadap kebu


In [29]:
from trl import SFTTrainer, SFTConfig

# mendefinisikan konfigurasi menggunakan SFTConfig
args = SFTConfig(
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    output_dir = "outputs",
)

# menginisialisasi SFTTrainer
trainer = SFTTrainer(
    model = model,
    train_dataset = dataset_train,
    processing_class = tokenizer,
    args = args,
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/7 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.428539
2,1.428568
3,1.417607
4,1.377133
5,1.317173
6,1.241101
7,1.152610
8,1.073788
9,0.999526
10,0.925743


In [30]:
save_path = f"{output_dir}/lora_model_1b"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model LoRA berhasil disimpan di: {save_path}")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/junior-ai-engineer/data/lora_model_1b/tokenizer_config.json.


Model LoRA berhasil disimpan di: /content/drive/MyDrive/junior-ai-engineer/data/lora_model_1b


# **Demo**

In [31]:
from IPython.display import clear_output, display
import ipywidgets as widgets
import torch
from unsloth import FastLanguageModel

# - Muat Model dan Tokenizer dari Folder LoRA
max_seq_length = 2048
dtype = None  # Auto-detect Float16/Bfloat16
load_in_4bit = True

model_path = "/content/drive/MyDrive/junior-ai-engineer/data/lora_model"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_path,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Set mode inferensi (mempercepat generasi & hemat memory GPU)
FastLanguageModel.for_inference(model)

# - Template Prompt Alpaca (Harus persis sama seperti saat training)
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
"""

# - Elemen UI (Widgets)
instruction_widget = widgets.Textarea(
    value=(
        "Jelaskan atau jawab pertanyaan berikut berdasarkan dokumen yang"
        " diberikan:"
    ),
    description="Instruction:",
    placeholder="Masukkan instruksi atau pertanyaan...",
    layout=widgets.Layout(width="95%", height="70px"),
)

input_widget = widgets.Textarea(
    description="Input/Konteks:",
    placeholder=(
        "Opsional: Tempelkan cuplikan teks/halaman Permenkes di sini jika ada..."
    ),
    layout=widgets.Layout(width="95%", height="100px"),
)

btn_generate = widgets.Button(
    description=" Generate Jawaban", button_style="success", icon="play"
)

output_area = widgets.Output(
    layout=widgets.Layout(border="1px solid #ccc", padding="12px", width="95%")
)


# - Fungsi Generasi Jawaban
def generate_response(b):
  with output_area:
    clear_output()
    inst = instruction_widget.value.strip()
    inp = input_widget.value.strip()

    if not inst:
      print("⚠️ Instruksi/Pertanyaan tidak boleh kosong!")
      return

    print("⏳ Sedang memproses output...\n")

    try:
      # Format prompt sesuai input pengguna
      prompt = alpaca_prompt.format(inst, inp)

      # Tokenisasi input dan kirim ke GPU
      inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

      # Evaluasi tanpa simpan gradien
      with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            use_cache=True,
            temperature=0.7,  # Variasi kata lebih alami
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

      # Ambil hanya token baru hasil generasi (bukan prompt asli)
      input_length = inputs.input_ids.shape[-1]
      generated_tokens = outputs[0][input_length:]
      output_text = tokenizer.decode(
          generated_tokens, skip_special_tokens=True
      ).strip()

      print("=== HASIL JAWABAN MODEL ===")
      print(output_text)

    except Exception as e:
      print(f"❌ Terjadi kesalahan: {e}")


btn_generate.on_click(generate_response)

# - Tampilkan UI Interaktif
display(
    widgets.VBox([
        instruction_widget,
        input_widget,
        btn_generate,
        widgets.HTML(value="<b>Output Model:</b>"),
        output_area,
    ])
)

==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /content/drive/MyDrive/junior-ai-engineer/data/lora_model as a legacy tokenizer.
